<a href="https://colab.research.google.com/github/MariosZamparas/DITchatbot/blob/collab/ollama_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ollama terminal installation:
- apt-get install zstd
- curl -fsSL https://ollama.com/install.sh | sh
- ollama serve

In [ ]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,448 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current 

In [ ]:
!ollama serve &> ollama.log &

In [ ]:
!ollama pull gemma3:12b

In [ ]:
!ollama list

NAME          ID              SIZE      MODIFIED          
gemma3:12b    f4031aab637d    8.1 GB    About an hour ago    


In [ ]:
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-huggingface
!pip install langchain-chroma
!pip install pypdf
!pip install langchain
!pip install langchain-ollama
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
#Ingestion script (only works if there is no vectorstore)

OLLAMA_MODEL = "gemma3:12b"

PDF_PATH = "/content/drive/MyDrive/ptuxiakh_docs"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 200

EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v2-moe"
EMBEDDING_MODEL_KWARGS = {"trust_remote_code": True}

VECTORSTORE_PATH = "/content/drive/MyDrive/vectorstore"

TOP_K = 8

import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import TextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


class MarkdownTitleTextSplitter(TextSplitter):
    def split_text(self, text: str):
        sections = []
        current_section = []
        previous_line = ""
        skipLine = False

        for line in text.splitlines():
            if line == "":
                continue
            elif line.startswith("#"):
                current_section.append(previous_line)
                if current_section:
                    sections.append("\n".join(current_section))
                current_section = [line]
                skipLine = True
            elif line.startswith("**"):
                current_section.append(previous_line)
                if current_section:
                    sections.append("\n".join(current_section))
                current_section = [line]
                skipLine = True
            elif line.startswith("|"):
                current_section.append(previous_line)
                if current_section:
                    sections.append("\n".join(current_section))
                current_section = [line]
                skipLine = True
            elif line.startswith("=="):
                if current_section:
                    sections.append("\n".join(current_section))
                current_section = [previous_line]
            else:
                if not skipLine:
                    current_section.append(previous_line)
                else:
                    skipLine = False
            previous_line = line

        if current_section:
            sections.append("\n".join(current_section))

        return sections


def load_documents(docs_path):
    loader = DirectoryLoader(
        docs_path,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    documents = loader.load()
    print(f"Loaded {len(documents)} markdown files from '{docs_path}'")
    return documents


def chunk_documents(docs):
    splitter = MarkdownTitleTextSplitter()
    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    all_chunks = []

    for doc in docs:
        sections = splitter.split_text(doc.page_content)
        for section in sections:
            if not section.strip():
                continue
            if len(section) > CHUNK_SIZE * 2:
                sub_chunks = fallback_splitter.create_documents(
                    [section],
                    metadatas=[doc.metadata]
                )
                all_chunks.extend(sub_chunks)
            else:
                all_chunks.append(
                    Document(page_content=section, metadata=doc.metadata)
                )

    print(f"Split into {len(all_chunks)} chunks")
    return all_chunks


def embed_and_store(chunks):
    print(f"Embedding {len(chunks)} chunks...")

    embedding_model = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs=EMBEDDING_MODEL_KWARGS
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=VECTORSTORE_PATH
    )

    print(f"Stored {len(chunks)} chunks in vectorstore at '{VECTORSTORE_PATH}'")
    return vectorstore


def run_ingestion():
    if os.path.exists(VECTORSTORE_PATH):
        print("Vectorstore already exists. Delete it to re-ingest.")
        return

    docs = load_documents(PDF_PATH)
    chunks = chunk_documents(docs)
    embed_and_store(chunks)


run_ingestion()


/tmp/ipykernel_2665/3974088530.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loaded 572 markdown files from '/content/drive/MyDrive/ptuxiakh_docs'
Split into 20244 chunks
Embedding 20244 chunks...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/554 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.10k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Stored 20244 chunks in vectorstore at '/content/drive/MyDrive/vectorstore'


In [ ]:
import os

#config.py
OLLAMA_MODEL = "gemma3:12b"

"""
LLMs reccomendations:

https://ollama.com/ilsp/llama-krikri-8b-instruct
https://ollama.com/library/gemma3
https://ollama.com/library/gemma4

"""

PDF_PATH = "/content/drive/MyDrive/ptuxiakh_docs"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 200

EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v2-moe"
EMBEDDING_MODEL_KWARGS = {"trust_remote_code": True}

VECTORSTORE_PATH = "/content/drive/MyDrive/vectorstore"

TOP_K = 8

#imports
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage

#ingest.py
def load_vectorstore():
    embedding_model = HuggingFaceEmbeddings(
        model_name= EMBEDDING_MODEL,
        model_kwargs= EMBEDDING_MODEL_KWARGS
    )

    vectorstore = Chroma(
        persist_directory= VECTORSTORE_PATH,
        embedding_function=embedding_model
    )

    print("Vectorstore loaded successfully.")
    return vectorstore

def get_retriever(vectorstore):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": TOP_K}
    )
    return retriever

def retrieve_chunks(query, retriever):
    chunks = retriever.invoke(query)
    return chunks
"""
# Test
if __name__ == "__main__":
    vs = load_vectorstore()
    retriever = get_retriever(vs)
    results = retrieve_chunks("ποιά είναι τα μαθήματα του πρώτου εξαμήνου", retriever)

    for i, chunk in enumerate(results):
        print(f"\n--- Chunk {i+1} ---")
        print(chunk.page_content)
        print(f"Source: {chunk.metadata.get('source', 'unknown')}")
"""


#prompt.py
def build_prompt_template():
    prompt = ChatPromptTemplate.from_messages([

        #TODO: Rewrite prompt (better)

        ("system", """

        You are an academic assistant for the Department of Computer Science and Telecommunications at the University of Arta (Πανεπιστήμιο Ιωαννίνων, Τμήμα Πληροφορικής και Τηλεπικοινωνιών).

        RESPONSE LANGUAGE:
        Always respond in the same language as the user's query (input), regardless of the language of the retrieved context. If the query is in Greek, respond in Greek; if in English, respond in English.

        SCOPE:
        Only answer questions related to the department: academic programs, courses, schedules, faculty, admissions, regulations, facilities, and administrative procedures. If a question is unrelated to the department, politely decline and state that you only handle department-related questions.

        GROUNDING:
        Answer strictly based on the provided context below. Do not use prior knowledge or make assumptions beyond what is given. If the context does not contain enough information to answer confidently, respond with "Δεν φαίνεται να υπάρχουν διαθέσιμες πληροφορίες για αυτο το ερώτημα. Παρακαλώ επικοινωνήστε με την γραμματεία." (Or the same sentence in the languege of the query) rather than guessing.

        STYLE:
        Be concise and precise. Use a formal but approachable tone appropriate for academic communication. Avoid unnecessary repetition of the question. Make the answer seamless, as if you are not rdirectly refrencing something, unless specifically asked to cite a source.

        Context: {context}

"""),
        ("human", "{input}"),
    ])

    #TODO: add context to human

    return prompt

def format_context(chunks):
    formatted = ""
    for i, chunk in enumerate(chunks):
        source = chunk.metadata.get("source", "unknown")
        formatted += f"[{i+1}] (Source: {source})\n{chunk.page_content}\n\n"
    return formatted



#agent.py
def load_llm():
    llm = ChatOllama(model= OLLAMA_MODEL)
    return llm

def build_chain(prompt_template, llm):
    chain = prompt_template | llm
    return chain

def rewrite_query(query, llm):

    #TODO: Rewrite prompt (better)

    rewrite_prompt = rewrite_prompt = f"""You are a query rewriting assistant for a retrieval system indexing official documents from the Department of Computer Science and Telecommunications, University of Arta.

    Your task: rewrite the query below into a single, well-formed search query using formal academic terminology, as it would appear in an official university guide or regulation document. This improves semantic matching against the indexed documents.

    RULES:
    - Preserve the query's original language (Greek stays Greek, English stays English).
    - Preserve the original intent and scope exactly. Do not add specific facts, names, or numbers that are not implied by the original query.
    - Expand vague or colloquial phrasing into precise academic/administrative terminology.
    - Do not answer the question. Only rewrite it.
    - Return ONLY the rewritten query, with no explanation, labels, or quotation marks.

    Examples:

    Original: "what classes are in the first semester?"
    Rewritten: What is the curriculum for the first semester of the Department of Computer Science and Telecommunications study program?

    Original: "who teaches programming?"
    Rewritten: Which faculty member is responsible for teaching the Programming course in the Department of Computer Science and Telecommunications?

    Original: "how do I apply for an internship?"
    Rewritten: What is the procedure for applying for a practical training (internship) placement in the Department of Computer Science and Telecommunications?

    Original: "when are the exams?"
    Rewritten: What is the examination period schedule for the Department of Computer Science and Telecommunications?


    Original: "{query}"
    Rewritten:"""

    response = llm.invoke(rewrite_prompt)
    rewritten = response.content.strip()
    print(f"Rewritten query: {rewritten}")
    return rewritten

def ask(query, chain, retriever, llm, history):
    rewritten_query = rewrite_query(query, llm)
    chunks = retrieve_chunks(rewritten_query, retriever)
    context = format_context(chunks)

    print("\n===== CONTEXT SENT TO LLM =====")
    print(context)
    print("================================\n")

    response = chain.invoke({
        "input": query,
        "context": context,
        "history": history,
    })

    return response.content


#TODO: make an llm orchestrator class that can call multiple llms for different tasks (rewrite, answer, etc)

#main.py
def main():
    # Guard — make sure ingestion has been run first
    if not os.path.exists("/content/drive/MyDrive/vectorstore"):
        print("Vectorstore not found. Please run ingest.py first.")
        return

    print("Loading components...")

    vectorstore = load_vectorstore()
    retriever = get_retriever(vectorstore)
    prompt_template = build_prompt_template()
    llm = load_llm()
    chain = build_chain(prompt_template, llm)

    print("Ready! Type your question. Type '0' to exit.\n")

    history = []

    while True:
        user_input = input("You: ").strip()

        if user_input == "0":
            print("Goodbye!")
            break

        if not user_input:
            continue

        try:
            response = ask(user_input, chain, retriever, llm, history)

            history.append(HumanMessage(content=user_input))
            history.append(AIMessage(content=response))

            print(f"\nAssistant: {response}\n")

        except Exception as e:
            print(f"Error: {e}")
            break


if __name__ == "__main__":
    main()

#TODO: maybe implement langgraph

Loading components...


/root/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


Vectorstore loaded successfully.
Ready! Type your question. Type '0' to exit.

You: Tell me about the internship programme
Rewritten query: What are the objectives, eligibility criteria, duration, and assessment methods associated with the practical training (internship) program offered by the Department of Computer Science and Telecommunications?

===== CONTEXT SENT TO LLM =====
[1] (Source: /content/drive/MyDrive/ptuxiakh_docs/www.dit.uoi.gr_files_b20_kanonismos_praktikis_askisis.pdf.md)
# 1. Γενικά 
Στο πρόγραμμα σπουδών Πληροφορικής και Τηλεπικοινωνιών η πρακτική άσκηση προσφέρεται μετά το 6ο εξάμηνο 
σε προαιρετική βάση, για χρονικό διάστημα 2 μηνών, και πραγματοποιείται τόσο στο δημόσιο όσο και στον ιδιωτικό 
τομέα, καθώς επίσης και σε επιχειρήσεις τ ης Ευρωπαϊκής Ένωσης στα πλαίσια ευρωπαϊκών προγραμμάτων. Για να 
συμμετάσχει ο φοιτητής θα πρέπει να έχει προηγουμένως επιτύχει σε τουλάχιστον 24 υποχρεωτικά μαθήματα. 
Η πρακτική άσκηση στοχεύει στο να εισάγει το φοιτητή στο επαγγε

KeyboardInterrupt: Interrupted by user

In [ ]:
#Chunk inspector

#config.py
OLLAMA_MODEL = "gemma3"

"""
LLMs reccomendations:

https://ollama.com/ilsp/llama-krikri-8b-instruct
https://ollama.com/library/gemma3
https://ollama.com/library/gemma4

"""

PDF_PATH = "/content/drive/MyDrive/ptuxiakh_docs"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 200

EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v2-moe"
EMBEDDING_MODEL_KWARGS = {"trust_remote_code": True}

VECTORSTORE_PATH = "/content/drive/MyDrive/vectorstore"

TOP_K = 8

def inspect_chunks():
    embedding_model = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs=EMBEDDING_MODEL_KWARGS
    )

    vectorstore = Chroma(
        persist_directory=VECTORSTORE_PATH,
        embedding_function=embedding_model
    )

    data = vectorstore.get()

    documents = data["documents"]
    metadatas = data["metadatas"]

    print(f"Total chunks in vectorstore: {len(documents)}\n")
    print("=" * 60)

    for i, (chunk, metadata) in enumerate(zip(documents, metadatas)):
        source = metadata.get("source", "unknown")
        print(f"\n[Chunk {i+1}] Source: {source}")
        print("-" * 40)
        print(chunk)
        print("=" * 60)


def inspect_chunks_target(filter_source=None):
    embedding_model = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs=EMBEDDING_MODEL_KWARGS
    )

    vectorstore = Chroma(
        persist_directory=VECTORSTORE_PATH,
        embedding_function=embedding_model
    )

    data = vectorstore.get()
    documents = data["documents"]
    metadatas = data["metadatas"]

    # Φιλτράρισμα αν δοθεί όνομα αρχείου
    pairs = [
        (chunk, meta) for chunk, meta in zip(documents, metadatas)
        if filter_source is None or filter_source in meta.get("source", "")
    ]

    print(f"Showing {len(pairs)} / {len(documents)} total chunks\n")
    print("=" * 60)

    for i, (chunk, metadata) in enumerate(pairs):
        source = metadata.get("source", "unknown")
        print(f"\n[Chunk {i+1}] Source: {source}")
        print("-" * 40)
        print(chunk)
        print("=" * 60)


inspect_chunks_target("odigos_spoudon_202122v2")

/root/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


Showing 1666 / 1718 total chunks


[Chunk 1] Source: /content/drive/MyDrive/ptuxiakh_docs/odigos_spoudon_202122v2.md
----------------------------------------

![University of Ioannina logo](page_1_image_2_v2.jpg)

[Chunk 2] Source: /content/drive/MyDrive/ptuxiakh_docs/odigos_spoudon_202122v2.md
----------------------------------------
# <u>ΠΑΝΕΠΙΣΤΗΜΙΟ ΙΩΑΝΝΙΝΩΝ</u>
![D.i&t logo](page_1_image_1_v2.jpg)

[Chunk 3] Source: /content/drive/MyDrive/ptuxiakh_docs/odigos_spoudon_202122v2.md
----------------------------------------
## <u>ΣΧΟΛΗ ΠΛΗΡΟΦΟΡΙΚΗΣ ΚΑΙ ΤΗΛΕΠΙΚΟΙΝΩΝΙΩΝ</u>
## <u>ΣΧΟΛΗ ΠΛΗΡΟΦΟΡΙΚΗΣ ΚΑΙ ΤΗΛΕΠΙΚΟΙΝΩΝΙΩΝ</u>

[Chunk 4] Source: /content/drive/MyDrive/ptuxiakh_docs/odigos_spoudon_202122v2.md
----------------------------------------
## <u>ΤΜΗΜΑ ΠΛΗΡΟΦΟΡΙΚΗΣ ΚΑΙ ΤΗΛΕΠΙΚΟΙΝΩΝΙΩΝ</u>
## <u>ΤΜΗΜΑ ΠΛΗΡΟΦΟΡΙΚΗΣ ΚΑΙ ΤΗΛΕΠΙΚΟΙΝΩΝΙΩΝ</u>

[Chunk 5] Source: /content/drive/MyDrive/ptuxiakh_docs/odigos_spoudon_202122v2.md
----------------------------------------
# ΟΔΗΓΟΣ ΠΡΟΠΤΥΧΙΑΚΩΝ ΣΠΟΥ